# Fine-tune Laya to call MON's next move (Kaggle, GPU T4 x2)

Trains [Laya](https://huggingface.co/convaiinnovations/laya) on the blocks the trader logged with `STATE_LOG=true`. Each block is labelled with what the MON-USDC mid actually did `horizonBlocks` later: **buy** if it rose, **sell** if it fell. The output is a checkpoint folder you point `laya/server.py` at.

**Kaggle settings** (right sidebar, *Notebook options*):
* **Accelerator:** `GPU T4 x2` (one GPU works too, just slower)
* **Internet:** `On` (installs `laya`, downloads the base checkpoint)
* **Input:** *Add Input* → your dataset holding `states-*.jsonl` (or `.jsonl.gz`) and `question.json` from the bot's `data/laya/` folder

Kaggle gives about 30 GPU hours a week and 12 hours per session. A full run with the defaults takes a few hours; set `MAX_TRAIN = 10_000` and `EPOCHS = 1` for a first pass.

**What to expect.** Direction over the next 30 seconds is mostly noise. On held-out *later* data, beating the base rate by one to three points of accuracy with a lower log loss is a real result. Sixty percent or more almost always means something leaked. This notebook splits by time with an embargo and prints baselines next to the model so you can tell the difference.

## 1. GPUs

In [ ]:
import os, sys, json, glob, gzip, math, time, gc, shutil, subprocess
import numpy as np, pandas as pd, torch

IS_KAGGLE = os.path.exists("/kaggle/working")
SMOKE = os.environ.get("LAYA_SMOKE") == "1"  # tiny CPU run that exercises every cell; for testing the notebook itself
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")
if N_GPU == 0 and not SMOKE:
    raise RuntimeError("No GPU. On Kaggle: Notebook options -> Accelerator -> GPU T4 x2.")
if N_GPU == 1:
    print("One GPU: training runs without DDP and takes about twice as long.")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 2. Install
Pinned: the notebook uses Laya's own tokenization and training helpers, and `laya/requirements.txt` in the repo pins the same version for serving.

In [ ]:
if IS_KAGGLE:
    !pip install -q "laya==0.3.21"
import laya, transformers
print("laya", laya.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)

## 3. Settings
Defaults fit a Kaggle session (12 h limit) with room to spare. `STRIDE` thins the data: neighbouring blocks share 95% of their future window, so every block adds little beyond every fifth one.

In [ ]:
WORK = "/kaggle/working" if IS_KAGGLE else os.path.abspath("work")
DATA_GLOB = os.environ.get("LAYA_DATA_GLOB", "/kaggle/input/**/states-*.jsonl*")
BASE_REPO, BASE_SUBFOLDER = "convaiinnovations/laya", "typed-decisions"  # 1,024-token context, already tuned on JSON decisions
OUTPUT_DIR = f"{WORK}/laya-trader"

MIN_MOVE_BPS = 0.0          # drop blocks whose move is this small or smaller (0 drops exact ties only)
STRIDE = 5                  # keep every 5th block
SPLIT = (0.70, 0.15, 0.15)  # train / val (epoch selection + calibration) / test, in time order
EPOCHS = 3
MAX_TRAIN = 40_000          # 3 epochs is roughly 3 to 5 h on 2x T4 (the log prints an ETA); lower it for a first run
MICRO_BATCH, GRAD_ACCUM = 8, 4  # effective batch = 8 x GPUs x 4
LR_ENCODER, LR_HEAD = 2e-5, 1e-4
VAL_MAX = 4_000             # val items scored after every epoch
TEST_MAX = None             # cap the test split (None = all)
SEED = 20260929
if SMOKE:
    STRIDE, EPOCHS, MAX_TRAIN, MICRO_BATCH, GRAD_ACCUM, VAL_MAX, TEST_MAX = 1, 1, 8, 2, 1, 8, 16

os.makedirs(WORK, exist_ok=True)
DEVICE = "cuda" if N_GPU else "cpu"

## 4. Load the logged blocks

In [ ]:
files = sorted(glob.glob(DATA_GLOB, recursive=True))
assert files, f"No files match {DATA_GLOB}. Add your dataset with Add Input, or fix DATA_GLOB."
rows, bad = [], 0
for f in files:
    with (gzip.open if f.endswith(".gz") else open)(f, "rt", encoding="utf-8") as fh:
        for line in fh:
            if line.strip():
                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    bad += 1  # a line cut off when the bot stopped
                    continue
                if r["block"] % STRIDE:
                    r["state"] = None  # every block's mid is needed for labels, but only every STRIDE-th state
                rows.append(r)
df = pd.DataFrame(rows).sort_values(["block", "ts"]).drop_duplicates("block").reset_index(drop=True)
del rows
assert df["v"].nunique() == 1, f"mixed state versions {sorted(df['v'].unique())}: train on one LAYA_STATE_VERSION at a time"
H = int(df["state"].dropna().iloc[0]["horizonBlocks"])

q_files = sorted(glob.glob(os.path.join(os.path.dirname(DATA_GLOB), "question.json"), recursive=True))
assert q_files, "question.json not found next to the data. Upload the bot's data/laya/question.json with it."
QUESTION = json.load(open(q_files[0], encoding="utf-8"))
QUESTIONS = {"direction": QUESTION}
KEYS = list(QUESTION["criteria"])  # option order, which is also the target order
assert set(KEYS) == {"buy", "sell"}, KEYS

hours = (df["ts"].iloc[-1] - df["ts"].iloc[0]) / 3.6e6
gaps = int((np.diff(df["block"]) > 20).sum())
print(f"{len(df):,} blocks from {len(files)} file(s), {hours:.1f} h, horizon {H} blocks, {gaps} gaps over 20 blocks, {bad} bad lines")
print("question:", QUESTION["instructions"][:120], "...")

## 5. Label: did the mid go up?
The label comes from the first logged block at or after `block + H` (within 10 blocks; otherwise the row is dropped). Blocks where the mid did not move carry no direction and are dropped.

In [ ]:
blocks, mids = df["block"].to_numpy(), df["mid"].to_numpy()
j = np.searchsorted(blocks, blocks + H)
ok = j < len(df)
jj = np.where(ok, j, 0)
ok &= (blocks[jj] - (blocks + H)) <= 10
df["retBps"] = np.where(ok, (mids[jj] - mids) / mids * 1e4, np.nan)
df = df[ok & df["state"].notna().to_numpy()].reset_index(drop=True)
flat = (df["retBps"].abs() <= MIN_MOVE_BPS).mean()
df = df[df["retBps"].abs() > MIN_MOVE_BPS].reset_index(drop=True)
df["label"] = np.where(df["retBps"] > 0, "buy", "sell")
print(f"{len(df):,} labelled blocks | {flat:.1%} dropped as flat | up {(df['label'] == 'buy').mean():.1%}")
print(df["retBps"].describe(percentiles=[.05, .25, .5, .75, .95]).round(2).to_string())

## 6. Split by time, with an embargo
A random split leaks: two neighbouring blocks share almost the same future, so the model would be tested on answers it has already seen. Instead, train on the oldest data, pick the epoch and fit calibration on the next slice, and test on the newest. Rows whose label window reaches into the next slice are dropped.

In [ ]:
EMBARGO = 0 if SMOKE else H
n = len(df)
a, b = int(n * SPLIT[0]), int(n * (SPLIT[0] + SPLIT[1]))
train = df.iloc[:a][df["block"].iloc[:a] + EMBARGO < df["block"].iloc[a]]
val = df.iloc[a:b][df["block"].iloc[a:b] + EMBARGO < df["block"].iloc[b]]
test = df.iloc[b:]
if len(train) > MAX_TRAIN:
    train = train.iloc[np.linspace(0, len(train) - 1, MAX_TRAIN).astype(int)]
val = val.iloc[np.linspace(0, len(val) - 1, min(len(val), VAL_MAX)).astype(int)] if len(val) else val
if TEST_MAX:
    test = test.iloc[:TEST_MAX]
for name, s in [("train", train), ("val", val), ("test", test)]:
    span = (s["ts"].iloc[-1] - s["ts"].iloc[0]) / 3.6e6 if len(s) else 0
    print(f"{name:5s} {len(s):7,} rows  {span:5.1f} h  up {(s['label'] == 'buy').mean():.1%}")
assert len(train) and len(val) and len(test), "not enough data for three splits; collect for longer"
noise = 1.96 * math.sqrt(0.25 / len(test)) * 100
print(f"\nWith {len(test):,} test rows, accuracy differences under about ±{noise:.1f} points are noise (95%).")

## 7. Baselines: what the model has to beat
* **base rate**: always predict the training set's up share. Its log loss is the bar a real signal has to clear.
* **momentum / taker flow**: follow the last 20 blocks' return, or the taker imbalance.
* **mock heuristic**: the bot's stand-in model (`MockModel`) without its noise.

`edge` is the average move, in bps, you would have caught by going with the call, before spread and gas. `edge top 20%` counts only the 20% of calls the model was most confident about.

In [ ]:
from laya.common import ece_score

def evaluate(name, p_buy, split, probabilistic=True):
    y = (split["label"] == "buy").to_numpy()
    ret = split["retBps"].to_numpy()
    p = np.clip(np.asarray(p_buy, dtype=float), 1e-6, 1 - 1e-6)
    call = p >= 0.5
    conf = np.maximum(p, 1 - p)
    signed = np.where(call, ret, -ret)
    top = conf >= np.quantile(conf, 0.8)
    out = {"model": name, "accuracy": (call == y).mean(), "edge bps": signed.mean(), "edge top 20% bps": signed[top].mean()}
    if probabilistic:
        out["log loss"] = -(y * np.log(p) + (~y) * np.log(1 - p)).mean()
        out["brier"] = ((p - y) ** 2).mean()
        out["ECE"] = ece_score(conf, (call == y).astype(float))
    return out

col = lambda s, f: np.array([f(st) for st in s["state"]])
base_up = (train["label"] == "buy").mean()
results = [
    evaluate("base rate", np.full(len(test), base_up), test),
    evaluate("momentum (last 20 blocks)", 0.5 + 0.01 * np.sign(col(test, lambda s: s["returnsBps"]["last20"])), test, False),
    evaluate("taker flow (cvd)", 0.5 + 0.01 * np.sign(col(test, lambda s: s["flow"]["cvdMon"])), test, False),
]
def mock_signal(s):
    f = s["flow"]; vol = f["buyMon"] + f["sellMon"]
    return s["returnsBps"]["last20"] / 8 + s["imbalance"] * 1.5 + (f["cvdMon"] / vol if vol else 0) * 2
results.append(evaluate("mock heuristic", 1 / (1 + np.exp(-col(test, mock_signal))), test))
pd.DataFrame(results).round(4)

## 8. Tokenize, exactly the way the server will
The items are built with Laya's own `build_sequence` and the same question normalisation `Agent.predict` applies, so training sees the same tokens that inference will. Section 9 checks this against the real agent.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from laya.agent import Agent, _fix_tokenizer_config
from laya.common import build_sequence, encode_text, serialize_state, QTYPES

BASE_DIR = os.path.join(snapshot_download(BASE_REPO, allow_patterns=[f"{BASE_SUBFOLDER}/*"]), BASE_SUBFOLDER)
_fix_tokenizer_config(BASE_DIR)
tok = AutoTokenizer.from_pretrained(os.path.join(BASE_DIR, "tokenizer"))
cfg = json.load(open(os.path.join(BASE_DIR, "rl_agent_config.json")))
MAX_LEN, HEAD_MAX_LEN = cfg["max_len"], cfg["head_max_len"]
Q = Agent._to_internal(QUESTION)

def encode(state):
    state_ids = encode_text(tok, serialize_state(state).replace(tok.mask_token, " "), add_special_tokens=False)["input_ids"]
    ids, markers = build_sequence(tok, state, Q, MAX_LEN, HEAD_MAX_LEN, state_ids=state_ids)
    assert len(markers) == len(KEYS)
    return ids, markers

def to_items(split):
    items = []
    for st, lab in zip(split["state"], split["label"]):
        ids, markers = encode(st)
        items.append({"ids": ids, "markers": markers, "qtype": QTYPES["choice"],
                      "target": [float(k == lab) for k in KEYS], "label": KEYS.index(lab)})
    return items

t0 = time.time()
train_items, val_items = to_items(train), to_items(val)
lens = np.array([len(it["ids"]) for it in train_items])
print(f"{len(train_items):,} train + {len(val_items):,} val items in {time.time() - t0:.0f}s | "
      f"tokens p50 {int(np.median(lens))}, p95 {int(np.percentile(lens, 95))}, max {lens.max()} of {MAX_LEN} | "
      f"cut off at the limit: {(lens >= MAX_LEN).mean():.1%}")
torch.save({"train": train_items, "val": val_items}, f"{WORK}/items.pt")

## 9. Zero-shot check, before training
Scores the test split with the untouched base checkpoint, loaded the way `laya/server.py` loads it. This gives the "before" number and confirms the training tokens match what the agent produces. It takes a few minutes, before you commit hours to training.

In [ ]:
def p_buy(agent, split, bs=64):
    res = agent.predict_batch(list(split["state"]), QUESTIONS, batch_size=bs)
    return np.array([r["answers"]["direction"]["probabilities"]["buy"] for r in res])

agent = laya.load(BASE_DIR, device=DEVICE)
for st in list(test["state"].iloc[:5]) + list(train["state"].iloc[:5]):
    assert encode(st)[0] == agent._encode_state(st, ["direction"], {"direction": Q})[0]["ids"], \
        "training tokens differ from what the agent feeds the model; do not train on these items"
print("tokenization matches inference")
t0 = time.time()
results.append(evaluate("Laya zero-shot", p_buy(agent, test), test))
print(f"scored {len(test):,} test rows in {time.time() - t0:.0f}s")
del agent; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(results).round(4)

## 10. Training script
This is Laya's RLCD recipe from the official Kaggle notebook: a policy-gradient term scored by proper scoring rules, plus soft cross-entropy, fp16, gradient checkpointing, and DDP across both T4s. Three changes for this data:
* **validation after every epoch:** noisy market labels overfit fast, so the best epoch by val log loss is kept, not the last one;
* **calibration on the val split:** fitted with the weights that were kept;
* **1 or 2 GPUs:** runs with or without DDP.

In [ ]:
TRAIN_PY = r"""
import os, sys, json, time, math, random, datetime, contextlib
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward

def collate(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    k = max(len(it["markers"]) for it in items)
    b = {"input_ids": torch.full((n, L), pad_id, dtype=torch.long), "attention_mask": torch.zeros((n, L), dtype=torch.long),
         "marker_pos": torch.zeros((n, k), dtype=torch.long), "marker_mask": torch.zeros((n, k), dtype=torch.bool),
         "target": torch.zeros((n, k)), "qtype": torch.tensor([it["qtype"] for it in items]),
         "label": torch.tensor([it["label"] for it in items])}
    for i, it in enumerate(items):
        b["input_ids"][i, :len(it["ids"])] = torch.tensor(it["ids"])
        b["attention_mask"][i, :len(it["ids"])] = 1
        b["marker_pos"][i, :len(it["markers"])] = torch.tensor(it["markers"])
        b["marker_mask"][i, :len(it["markers"])] = True
        b["target"][i, :len(it["target"])] = torch.tensor(it["target"])
    return b

def fit_temperature(z, t):
    # One temperature for the choice head, by LBFGS on log T (as Laya's own notebook does).
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(t * torch.log_softmax(z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.5, 5.0).item())  # Laya clamps to this range at load

def main():
    base_dir, out_dir, items_path, hp_path = sys.argv[1:5]
    hp = json.load(open(hp_path))
    cuda, ddp = torch.cuda.is_available(), int(os.environ.get("WORLD_SIZE", "1")) > 1
    if ddp:
        dist.init_process_group("nccl" if cuda else "gloo", timeout=datetime.timedelta(minutes=60))
    rank, world = (dist.get_rank(), dist.get_world_size()) if ddp else (0, 1)
    local = int(os.environ.get("LOCAL_RANK", "0"))
    if cuda:
        torch.cuda.set_device(local)
    device = torch.device("cuda", local) if cuda else torch.device("cpu")
    barrier = dist.barrier if ddp else (lambda: None)
    log = (lambda *a: print(*a, flush=True)) if rank == 0 else (lambda *a: None)
    torch.manual_seed(hp["seed"])

    cfg = json.load(open(os.path.join(base_dir, "rl_agent_config.json")))
    tok = AutoTokenizer.from_pretrained(os.path.join(base_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(base_dir, "encoder"))
    model.load_state_dict(load_file(os.path.join(base_dir, "model.safetensors")), strict=True)
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device).train()
    net = DDP(model, device_ids=[local] if cuda else None, find_unused_parameters=True) if ddp else model

    data = torch.load(items_path, weights_only=False)
    train, val = data["train"], data["val"]
    E, MB, GA, G = hp["epochs"], hp["micro_batch"], hp["grad_accum"], 4
    if ddp:  # every rank must run the same number of backward passes, or the gradient all-reduce hangs
        train = train[: max(world * MB, len(train) // (world * MB) * (world * MB))]
    mine = train[rank::world]
    enc = [p for n, p in net.named_parameters() if "encoder." in n]
    head = [p for n, p in net.named_parameters() if "encoder." not in n]
    opt = torch.optim.AdamW([{"params": enc, "lr": hp["lr_encoder"]}, {"params": head, "lr": hp["lr_head"]}], weight_decay=0.01)
    updates = max(1, math.ceil(len(mine) / (MB * GA)) * E)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=updates, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=cuda)
    amp = lambda: torch.autocast(device.type, dtype=torch.float16, enabled=cuda)

    def forward(m, b):
        with amp():
            logits, act = m(b["input_ids"].to(device), b["attention_mask"].to(device), b["marker_pos"].to(device),
                            b["marker_mask"].to(device), b["qtype"].to(device))
        return logits.float(), act

    def score_val():
        model.eval()
        zs, nll, right = [], 0.0, 0
        with torch.no_grad():
            for i in range(0, len(val), 16):
                b = collate(val[i:i + 16], tok.pad_token_id)
                z = forward(model, b)[0].masked_fill(~b["marker_mask"].to(device), -1e4)
                nll += -(b["target"].to(device) * torch.log_softmax(z, -1)).sum().item()
                right += (z.argmax(-1).cpu() == b["label"]).sum().item()
                zs.append(z[:, :2].cpu())
        model.train()
        return nll / len(val), right / len(val), torch.cat(zs)

    def save(temperature):
        os.makedirs(out_dir, exist_ok=True)
        save_file({k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}, os.path.join(out_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(out_dir, "encoder"))
        tok.save_pretrained(os.path.join(out_dir, "tokenizer"))
        c = dict(cfg, fine_tuned=True, model_name="laya-trader")
        c["temperature"] = [temperature] + list(cfg["temperature"][1:])
        c.pop("temperature_by_options", None)  # older bucket values would override the new fit at inference
        json.dump(c, open(os.path.join(out_dir, "rl_agent_config.json"), "w"), indent=2)

    report = {"epochs": [], "train_items": len(train), "val_items": len(val), "gpus": world}
    best, best_z = float("inf"), None
    if rank == 0:
        nll0, acc0, _ = score_val()
        report["zero_shot_val"] = {"log_loss": nll0, "accuracy": acc0}
        log(f"zero-shot val: log loss {nll0:.4f}, accuracy {acc0:.3f}")
    barrier()
    log(f"training {len(train):,} items on {world} {'GPU' if cuda else 'CPU'} process(es), {E} epochs, {updates} updates")
    t0, step = time.time(), 0
    for epoch in range(E):
        random.Random(hp["seed"] + epoch + rank).shuffle(mine)
        sigma = 0.4 + (0.1 - 0.4) * (epoch / max(1, E - 1))
        opt.zero_grad(set_to_none=True)
        run_loss, n = 0.0, 0
        for i in range(0, len(mine), MB):
            b = collate(mine[i:i + MB], tok.pad_token_id)
            step_now = (n + 1) % GA == 0 or i + MB >= len(mine) or (hp.get("max_steps") and n + 1 >= hp["max_steps"])
            with (net.no_sync() if ddp and not step_now else contextlib.nullcontext()):  # all-reduce once per update, not per micro-batch
                logits, act = forward(net, b)
                mask, target = b["marker_mask"].to(device), b["target"].to(device)
                k = mask.sum(-1, keepdim=True).float()
                eps = torch.randn((G,) + logits.shape, device=device) * sigma * mask
                eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
                z = logits.detach().unsqueeze(0) + eps
                with torch.no_grad():
                    r = proper_reward(torch.softmax(z.masked_fill(~mask, -1e4), -1), target.unsqueeze(0), b["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                    adv = (r - r.mean(0, keepdim=True)) / ((r - r.mean(0, keepdim=True)).std() + 1e-6)
                logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
                ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
                loss = (-(adv * logp).mean() + ce) / GA + 0.0 * act.sum()
                scaler.scale(loss).backward()
            n += 1
            run_loss += ce.item()
            if step_now:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step()
                opt.zero_grad(set_to_none=True)
                step += 1
                if step % 50 == 0:
                    eta = (time.time() - t0) / step * (updates - step)
                    log(f"  epoch {epoch + 1}/{E} update {step}/{updates} | train CE {run_loss / n:.4f} | {time.time() - t0:.0f}s, ETA {eta / 60:.0f} min")
            if hp.get("max_steps") and n >= hp["max_steps"]:
                break
        barrier()
        if rank == 0:
            nll, acc, z = score_val()
            kept = nll < best
            if kept:
                best, best_z = nll, z
                save(cfg["temperature"][0])  # temperature is refit below; saving now survives a timeout
            report["epochs"].append({"epoch": epoch + 1, "train_ce": run_loss / max(1, n), "val_log_loss": nll, "val_accuracy": acc, "kept": kept})
            log(f"epoch {epoch + 1}: train CE {run_loss / max(1, n):.4f} | val log loss {nll:.4f}, accuracy {acc:.3f}{'  (kept)' if kept else ''} | {time.time() - t0:.0f}s")
        barrier()

    if rank == 0:
        t = torch.tensor([[it["target"][0], it["target"][1]] for it in val])
        temperature = fit_temperature(best_z, t) if len(val) >= 10 else 1.0
        save(temperature)
        report.update(best_val_log_loss=best, temperature=temperature, beat_zero_shot=best < report["zero_shot_val"]["log_loss"])
        json.dump(report, open(os.path.join(out_dir, "train_report.json"), "w"), indent=2)
        log(f"saved {out_dir} | best val log loss {best:.4f} (zero-shot {report['zero_shot_val']['log_loss']:.4f}) | temperature {temperature:.3f}")
    if ddp:
        dist.destroy_process_group()

if __name__ == "__main__":
    main()
"""
open(f"{WORK}/train.py", "w").write(TRAIN_PY)
print("wrote", f"{WORK}/train.py")

## 11. Train
Progress prints every 50 updates with an ETA. The best epoch is written to `OUTPUT_DIR` as soon as it is scored, so a session timeout keeps what finished.

In [ ]:
hp = {"epochs": EPOCHS, "micro_batch": MICRO_BATCH, "grad_accum": GRAD_ACCUM, "lr_encoder": LR_ENCODER,
      "lr_head": LR_HEAD, "seed": SEED, "max_steps": 2 if SMOKE else None}
json.dump(hp, open(f"{WORK}/hparams.json", "w"))
args = [BASE_DIR, OUTPUT_DIR, f"{WORK}/items.pt", f"{WORK}/hparams.json"]
if N_GPU > 1:
    cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={N_GPU}", f"{WORK}/train.py", *args]
else:
    cmd = [sys.executable, f"{WORK}/train.py", *args]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise RuntimeError(f"training exited with code {proc.returncode}; see the log above")

## 12. Test: fine-tuned against everything else
The test split is the newest data, never seen in training, epoch selection or calibration. The latency line times single decisions the way the bot asks for them. The bot has about 200 ms of its 300 ms block left for the model.

In [ ]:
agent = laya.load(OUTPUT_DIR, device=DEVICE)
ft = p_buy(agent, test)
results.append(evaluate("Laya fine-tuned", ft, test))
table = pd.DataFrame(results).round(4)

sample = list(test["state"].iloc[:30])
agent.system_one(sample[0], QUESTIONS)
lat = []
for st in sample:
    t0 = time.perf_counter(); agent.system_one(st, QUESTIONS); lat.append((time.perf_counter() - t0) * 1000)
print(f"single-decision latency on {DEVICE}: p50 {np.median(lat):.0f} ms, p95 {np.percentile(lat, 95):.0f} ms")

conf = np.maximum(ft, 1 - ft)
y = (test["label"] == "buy").to_numpy()
buckets = pd.qcut(conf, q=min(5, len(np.unique(conf))), duplicates="drop")
print("\naccuracy by confidence (a calibrated model's accuracy tracks its confidence):")
print(pd.DataFrame({"confidence": conf, "right": (ft >= 0.5) == y}).groupby(buckets, observed=True)
      .agg(rows=("right", "size"), mean_confidence=("confidence", "mean"), accuracy=("right", "mean")).round(3).to_string())
del agent; gc.collect(); torch.cuda.empty_cache()
table

## 13. Save and download
Writes `question.json` and `report.json` next to the weights and zips the folder (about 850 MB).
* **Interactive session:** download `laya-trader.zip` from the *Output* panel (`/kaggle/working`) before the session ends.
* **Save Version → Save & Run All:** runs the whole notebook in the background; the zip then stays in the version's *Output* tab.

Then, in the repo: unzip to `./laya-trader` and run `LAYA_CHECKPOINT=./laya-trader python laya/server.py`.

In [ ]:
json.dump(QUESTION, open(f"{OUTPUT_DIR}/question.json", "w"), indent=2)
report = {"results": table.to_dict("records"), "horizon_blocks": H, "min_move_bps": MIN_MOVE_BPS, "stride": STRIDE,
          "rows": {"train": len(train), "val": len(val), "test": len(test)},
          "test_span_utc": [pd.to_datetime(test["ts"].iloc[0], unit="ms").isoformat(), pd.to_datetime(test["ts"].iloc[-1], unit="ms").isoformat()],
          "train_report": json.load(open(f"{OUTPUT_DIR}/train_report.json")), "hparams": hp}
json.dump(report, open(f"{OUTPUT_DIR}/report.json", "w"), indent=2, default=float)
zip_path = shutil.make_archive(f"{WORK}/laya-trader", "zip", OUTPUT_DIR)
print(f"{zip_path}: {os.path.getsize(zip_path) / 1e6:.0f} MB")

## 14. Optional: push to a private Hugging Face repo
Add a **write** token as a Kaggle secret named `HF_TOKEN` (*Add-ons → Secrets*), set `HF_REPO`, and run. The server can then load it with `LAYA_CHECKPOINT=<HF_REPO> HF_TOKEN=<read token> python laya/server.py`.

In [ ]:
HF_REPO = ""  # e.g. "your-name/laya-trader"; empty skips this cell
if HF_REPO:
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(HF_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=OUTPUT_DIR, repo_id=HF_REPO, commit_message="laya-trader fine-tune")
    print(f"pushed to https://huggingface.co/{HF_REPO} (private)")